#  FIFA World Cup 2026 — Feature Engineering


| # | Feature Group | Description |
|---|--------------|-------------|
| 1 | **ELO Rating** | Dynamic skill score updated after every match since 1990 |
| 2 | **Rolling Form** | Win rate & goal stats over last 5 and 10 matches |
| 3 | **Head-to-Head** | Historical record between each pair of WC 2026 teams |
| 4 | **FIFA Ranking** | Current rank & points attached to each match |
| 5 | **ML Feature Matrix** | One row per match, labeled, ready for training |

## 0. Setup

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('')
RAW_DIR  = BASE_DIR / 'data' / 'raw'
PROC_DIR = BASE_DIR / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

# ── Load raw data ────────────────────────────────────────────────
df_results = pd.read_csv(RAW_DIR / 'results.csv', parse_dates=['date'])
df_rankings = pd.read_csv(RAW_DIR / 'fifa_rankings_2026.csv')
df_wc2026_teams = pd.read_csv(RAW_DIR / 'wc_2026_teams.csv')
df_fixtures = pd.read_csv(RAW_DIR / 'wc_2026_fixtures.csv', parse_dates=['date'])

# Drop future matches (no scores yet) for training data
df_played = df_results.dropna(subset=['home_score', 'away_score']).copy()
df_played = df_played[df_played['date'] >= '1990-01-01'].copy()
df_played = df_played.sort_values('date').reset_index(drop=True)

print(f'✅ Loaded {len(df_played):,} completed matches for feature engineering')
print(f'   Date range: {df_played["date"].min().date()} → {df_played["date"].max().date()}')

✅ Loaded 32,286 completed matches for feature engineering
   Date range: 1990-01-12 → 2026-06-10


---
## 1. ELO Rating System

ELO is a dynamic rating: every match updates both teams' scores based on the result vs expected outcome.

**Formula:**  
`new_rating = old_rating + K × (actual - expected)`

- `expected = 1 / (1 + 10^((opp_elo - team_elo) / 400))`
- `actual` = 1 (win), 0.5 (draw), 0 (loss)
- `K` = importance weight: World Cup = 60, WC qualifier = 40, friendly = 20

In [18]:
import unicodedata

def _norm(s: str) -> str:
    s = unicodedata.normalize('NFKD', str(s).lower())
    return ''.join(c for c in s if not unicodedata.combining(c))


def get_k_factor(tournament: str, date: pd.Timestamp) -> int:

    t = _norm(tournament)

    
    MAJOR = ['fifa world cup', 'uefa euro', 'copa america',
         'african cup of nations', 'afc asian cup', 'gold cup',
         'nations league', 'confederations cup', 'olympic']

    is_qual  = 'qualification' in t or 'qualifier' in t
    is_major = any(x in t for x in MAJOR)

    if t == 'fifa world cup':          # finals only
        return 60
    if is_qual and date >= pd.Timestamp('2025-01-01'):
        return 30 if is_major else 15  # minor qualifiers match their finals
    if is_major:
        return 45
    if 'friendly' in t:
        return 20
    return 10


In [19]:

def compute_elo(df: pd.DataFrame,
                start_elo: int = 100,
                home_advantage: int = 100) -> tuple[pd.DataFrame, dict]:
    """
    Walk through every match chronologically and compute ELO.
    Returns:
      - df with columns home_elo_before, away_elo_before, home_elo_after, away_elo_after
      - final_elos dict: {team: current_elo}
    """
    elos = defaultdict(lambda: start_elo)

    home_elo_before, away_elo_before = [], []
    home_elo_after,  away_elo_after  = [], []

    for _, row in df.iterrows():
        h, a = row['home_team'], row['away_team']
        eh, ea = elos[h], elos[a]

        home_elo_before.append(eh)
        away_elo_before.append(ea)

        # Add home advantage unless neutral venue
        eh_adj = eh + (0 if row['neutral'] else home_advantage)

        # Expected scores
        exp_h = 1 / (1 + 10 ** ((ea - eh_adj) / 400))
        exp_a = 1 - exp_h

        # Actual scores
        hs, as_ = row['home_score'], row['away_score']
        if hs > as_:   act_h, act_a = 1.0, 0.0
        elif hs < as_: act_h, act_a = 0.0, 1.0
        else:          act_h, act_a = 0.5, 0.5

        K = get_k_factor(row['tournament'], row['date'])

        elos[h] = eh + K * (act_h - exp_h)
        elos[a] = ea + K * (act_a - exp_a)

        home_elo_after.append(elos[h])
        away_elo_after.append(elos[a])

    df = df.copy()
    df['home_elo_before'] = home_elo_before
    df['away_elo_before'] = away_elo_before
    df['home_elo_after']  = home_elo_after
    df['away_elo_after']  = away_elo_after
    df['elo_diff']        = df['home_elo_before'] - df['away_elo_before']

    return df, dict(elos)


print(f'⚙️  Computing ELO ratings across {len(df_played):,} matches...')
df_played, current_elos = compute_elo(df_played)
print(f'✅ Done')

# Save current ELO snapshot
df_elo_snapshot = (pd.DataFrame.from_dict(current_elos, orient='index', columns=['elo'])
                     .rename_axis('team')
                     .reset_index()
                     .sort_values('elo', ascending=False)
                     .reset_index(drop=True))
df_elo_snapshot.index += 1
df_elo_snapshot.to_csv(PROC_DIR / 'elo_all_teams.csv')

print('\n Top 20 teams by ELO (current):')
display(df_elo_snapshot.head(20))

⚙️  Computing ELO ratings across 32,286 matches...
✅ Done

 Top 20 teams by ELO (current):


,team,elo
1,Spain,653.736717
2,Argentina,645.164594
3,France,579.438462
4,Brazil,545.661594
5,Portugal,538.987762
6,England,520.990017
7,Colombia,514.775285
8,Ecuador,513.501442
9,Mexico,487.649822
10,Japan,487.097583


In [20]:
# ELO for all 48 WC 2026 teams
wc26_teams = df_wc2026_teams['team'].tolist()
wc26_elos = [(t, round(current_elos.get(t, 100), 1)) for t in wc26_teams]
df_wc26_elo = (pd.DataFrame(wc26_elos, columns=['team', 'elo'])
                 .sort_values('elo', ascending=False)
                 .reset_index(drop=True))
df_wc26_elo.index += 1

print('ELO ratings for all 48 WC 2026 teams:')
display(df_wc26_elo)

ELO ratings for all 48 WC 2026 teams:


,team,elo
1,Spain,653.7
2,Argentina,645.2
3,France,579.4
4,Brazil,545.7
5,Portugal,539.0
6,England,521.0
7,Ecuador,513.5
8,Mexico,487.6
9,Japan,487.1
10,Germany,483.6


---
## 2. Rolling Form Features

For each team, compute rolling stats over their **last 5** and **last 10** matches prior to each game:
- Win rate
- Goals scored per game (avg)
- Goals conceded per game (avg)
- Goal difference per game (avg)

In [21]:
def compute_rolling_form(df: pd.DataFrame, windows: list[int] = [3, 5, 10]) -> pd.DataFrame:
    """
    For each match row, look back N matches for each team and compute form stats.
    Uses merge_asof for efficient lookup instead of iterrows.
    """
    # Build unified per-team timeline
    def make_records(df, side):
        other = 'away' if side == 'home' else 'home'
        rec = df[['date', f'{side}_team', f'{other}_team',
                   f'{side}_score', f'{other}_score']].copy()
        rec.columns = ['date', 'team', 'opponent', 'gf', 'ga']
        rec['win']  = (rec['gf'] > rec['ga']).astype(int)
        rec['draw'] = (rec['gf'] == rec['ga']).astype(int)
        rec['gd']   = rec['gf'] - rec['ga']   # raw gd before rolling
        return rec

    team_history = pd.concat([
        make_records(df, 'home'),
        make_records(df, 'away')
    ]).sort_values(['team', 'date']).reset_index(drop=True)

    # Compute rolling stats per team, shift(1) to avoid data leakage
    stat_frames = []
    for team, grp in team_history.groupby('team'):
        grp = grp.sort_values('date').reset_index(drop=True)
        out = grp[['date', 'team']].copy()
        for w in windows:
            rolled = grp[['win', 'gf', 'ga', 'gd']].shift(1).rolling(w, min_periods=1)
            out[f'win_rate_{w}'] = rolled['win'].mean()
            out[f'avg_gf_{w}']   = rolled['gf'].mean()
            out[f'avg_ga_{w}']   = rolled['ga'].mean()
            out[f'avg_gd_{w}']   = rolled['gd'].mean()  # roll raw gd, not derived
        stat_frames.append(out)

    all_stats = pd.concat(stat_frames).sort_values(['team', 'date']).reset_index(drop=True)

    stat_cols = [f'{s}_{w}' for w in windows
                             for s in ['win_rate', 'avg_gf', 'avg_ga', 'avg_gd']]

    # Use merge_asof per team for fast nearest-prior lookup (replaces iterrows)
    df = df.sort_values('date').reset_index(drop=True)

    for side in ['home', 'away']:
        merged_parts = []
        for team, grp in df.groupby(f'{side}_team'):
            team_form = all_stats[all_stats['team'] == team].sort_values('date')
            # merge_asof finds the last row in team_form strictly before match date
            merged = pd.merge_asof(
                grp.reset_index(),          # carries original index
                team_form[['date'] + stat_cols],
                on='date',
                direction='backward',
                allow_exact_matches=False   # strictly before match date
            )
            merged_parts.append(merged.set_index('index'))

        form_df = pd.concat(merged_parts).reindex(df.index)
        for col in stat_cols:
            df[f'{side}_{col}'] = form_df[col].values

    return df.sort_values('date').reset_index(drop=True)


print('⚙️  Computing rolling form features (this takes ~30 seconds)...')
df_played = compute_rolling_form(df_played)
print('✅ Rolling form computed')
print('New columns added:', [c for c in df_played.columns if 'win_rate' in c or 'avg_g' in c])

⚙️  Computing rolling form features (this takes ~30 seconds)...
✅ Rolling form computed
New columns added: ['home_win_rate_3', 'home_avg_gf_3', 'home_avg_ga_3', 'home_avg_gd_3', 'home_win_rate_5', 'home_avg_gf_5', 'home_avg_ga_5', 'home_avg_gd_5', 'home_win_rate_10', 'home_avg_gf_10', 'home_avg_ga_10', 'home_avg_gd_10', 'away_win_rate_3', 'away_avg_gf_3', 'away_avg_ga_3', 'away_avg_gd_3', 'away_win_rate_5', 'away_avg_gf_5', 'away_avg_ga_5', 'away_avg_gd_5', 'away_win_rate_10', 'away_avg_gf_10', 'away_avg_ga_10', 'away_avg_gd_10']


---
## 3. Head-to-Head Records

For every pair of WC 2026 teams, compute their all-time head-to-head record:
- Games played, wins, draws, losses
- Goals scored / conceded
- Win rate from each team's perspective

This is stored as a lookup table used during feature assembly.

In [22]:
def compute_h2h(df: pd.DataFrame, teams: list[str],
                ref_date: pd.Timestamp = pd.Timestamp('2026-01-01'),
                decay: float = 0.1) -> pd.DataFrame:
    rows = []

    for i, team_a in enumerate(teams):
        for team_b in teams[i+1:]:
            mask = (
                ((df['home_team'] == team_a) & (df['away_team'] == team_b)) |
                ((df['home_team'] == team_b) & (df['away_team'] == team_a))
            )
            h2h = df[mask].copy()

            if len(h2h) == 0:
                for ta, tb in [(team_a, team_b), (team_b, team_a)]:
                    rows.append({
                        'team_a': ta, 'team_b': tb, 'games': 0,
                        'a_wins': 0, 'draws': 0, 'b_wins': 0,
                        'a_goals': 0.0, 'b_goals': 0.0,
                        'avg_goal_diff': 0.0,
                        'a_win_rate': np.nan,           # honest: no data
                        'a_win_rate_weighted': np.nan,
                    })
                continue

            a_home = (h2h['home_team'] == team_a).values
            a_scored = np.where(a_home, h2h['home_score'].values, h2h['away_score'].values)
            b_scored = np.where(a_home, h2h['away_score'].values, h2h['home_score'].values)

            a_wins = int((a_scored > b_scored).sum())
            b_wins = int((a_scored < b_scored).sum())
            draws  = int((a_scored == b_scored).sum())
            games  = len(h2h)

            a_goals = float(a_scored.sum())
            b_goals = float(b_scored.sum())

            a_win_rate = (a_wins + 0.5 * draws) / games

            # Recency-weighted win rate
            years_ago = (ref_date - h2h['date']).dt.days.values / 365.25
            w = np.exp(-decay * years_ago)
            a_win_mask   = a_scored > b_scored
            draw_mask    = a_scored == b_scored
            a_win_rate_w = ((w * a_win_mask).sum() + 0.5 * (w * draw_mask).sum()) / w.sum()

            for ta, tb, ag, bg, aw, bw, wr, wrw in [
                (team_a, team_b, a_goals, b_goals, a_wins, b_wins,
                 a_win_rate, a_win_rate_w),
                (team_b, team_a, b_goals, a_goals, b_wins, a_wins,
                 1 - a_win_rate, 1 - a_win_rate_w),
            ]:
                rows.append({
                    'team_a': ta, 'team_b': tb, 'games': games,
                    'a_wins': aw, 'draws': draws, 'b_wins': bw,
                    'a_goals': ag, 'b_goals': bg,
                    'avg_goal_diff': round((ag - bg) / games, 3),
                    'a_win_rate': round(wr, 3),
                    'a_win_rate_weighted': round(wrw, 3),
                })

    return pd.DataFrame(rows)


print('⚙️  Computing head-to-head records for all 48 WC 2026 teams...')
df_h2h = compute_h2h(df_played, wc26_teams)
df_h2h.to_csv(PROC_DIR / 'h2h_records.csv', index=False)
print(f'✅ H2H records: {len(df_h2h)} pairings saved')

# Show some interesting matchups
print('\nMost-played rivalries among WC 2026 teams:')
display(df_h2h.sort_values('games', ascending=False).head(10))

print('\nPairings with no historical record (debut matchups):')
print(df_h2h[df_h2h['games'] == 0][['team_a','team_b']].to_string())

⚙️  Computing head-to-head records for all 48 WC 2026 teams...
✅ H2H records: 2256 pairings saved

Most-played rivalries among WC 2026 teams:


,team_a,team_b,games,a_wins,draws,b_wins,a_goals,b_goals,avg_goal_diff,a_win_rate,a_win_rate_weighted
22,Mexico,United States,50,16,13,21,52.0,63.0,-0.220,0.450,0.452
23,United States,Mexico,50,21,13,16,63.0,52.0,0.220,0.550,0.548
750,Brazil,Argentina,40,16,10,14,60.0,46.0,0.350,0.525,0.437
751,Argentina,Brazil,40,14,10,16,46.0,60.0,-0.350,0.475,0.563
222,South Korea,Japan,33,12,12,9,37.0,35.0,0.061,0.545,0.468
223,Japan,South Korea,33,9,12,12,35.0,37.0,-0.061,0.455,0.532
1364,Ecuador,Venezuela,31,15,6,10,54.0,32.0,0.710,0.581,0.548
1365,Venezuela,Ecuador,31,10,6,15,32.0,54.0,-0.710,0.419,0.452
1961,Argentina,Chile,31,19,11,1,49.0,17.0,1.032,0.790,0.819
1960,Chile,Argentina,31,1,11,19,17.0,49.0,-1.032,0.210,0.181



Pairings with no historical record (debut matchups):
                      team_a                  team_b
16                    Mexico                 Morocco
17                   Morocco                  Mexico
44                    Mexico                 Tunisia
45                   Tunisia                  Mexico
74                    Mexico                  Jordan
75                    Jordan                  Mexico
84                    Mexico              Cape Verde
85                Cape Verde                  Mexico
94              South Africa             South Korea
95               South Korea            South Africa
100             South Africa  Bosnia and Herzegovina
101   Bosnia and Herzegovina            South Africa
102             South Africa                   Qatar
103                    Qatar            South Africa
104             South Africa             Switzerland
105              Switzerland            South Africa
110             South Africa                 

---
## 4. Attach FIFA Rankings to Each Match

We use the current April 2026 rankings for WC 2026 predictions.
For the training dataset (historical matches), we use a rank proxy derived from ELO.

In [23]:
# ── Rank / ELO feature attachment ──────────────────────────────────────────
#
# Strategy:
#   Historical training rows  → ELO at match time (already in df_played if
#                                compute_elo() was called earlier)
#   WC 2026 fixture rows      → actual FIFA ranking snapshot (df_rankings)
#
# We never attach the 2026 snapshot to historical rows — that would be leakage.
# ───────────────────────────────────────────────────────────────────────────

# ── 1. Verify ELO columns exist on df_played (should be from earlier step) ──
ELO_COLS = ['home_elo_before', 'away_elo_before']   # adjust names if yours differ
missing_elo = [c for c in ELO_COLS if c not in df_played.columns]
if missing_elo:
    raise ValueError(
        f"ELO columns missing from df_played: {missing_elo}\n"
        "Run compute_elo() before this cell."
    )

# ── 2. Derive pseudo-rank from ELO for historical rows ──────────────────────
#
# Rank = ordinal position when teams sorted by ELO descending at that date.
# We do this per-match-date snapshot to stay strictly historical.
#
# Build a long table: one row per (date, team, elo_before)
home_elo = df_played[['date', 'home_team', 'home_elo_before']].rename(
    columns={'home_team': 'team', 'home_elo_before': 'elo'})
away_elo = df_played[['date', 'away_team', 'away_elo_before']].rename(
    columns={'away_team': 'team', 'away_elo_before': 'elo'})

elo_long = (
    pd.concat([home_elo, away_elo])
    .drop_duplicates(subset=['date', 'team'])
    .sort_values(['date', 'elo'], ascending=[True, False])
)

# Per date: assign pseudo-rank (1 = highest ELO)
elo_long['elo_pseudo_rank'] = (
    elo_long.groupby('date')['elo']
    .rank(ascending=False, method='min')
    .astype(int)
)

# Pivot back to per-match: attach pseudo-rank to home and away
elo_rank_lookup = elo_long.set_index(['date', 'team'])['elo_pseudo_rank']

df_played['home_elo_rank'] = df_played.set_index(
    ['date', 'home_team']).index.map(elo_rank_lookup)
df_played['away_elo_rank'] = df_played.set_index(
    ['date', 'away_team']).index.map(elo_rank_lookup)

# ── 3. WC 2026 fixture rows → attach actual FIFA rankings ───────────────────
rank_lookup   = dict(zip(df_rankings['team'], df_rankings['rank']))
points_lookup = dict(zip(df_rankings['team'], df_rankings['points']))

# These columns only make sense on 2026 fixtures, not historical rows
# They live on df_fixtures (or whatever holds your upcoming matches)
df_fixtures['home_fifa_rank']    = df_fixtures['home'].map(rank_lookup)
df_fixtures['away_fifa_rank']    = df_fixtures['away'].map(rank_lookup)
df_fixtures['home_fifa_points']  = df_fixtures['home'].map(points_lookup)
df_fixtures['away_fifa_points']  = df_fixtures['away'].map(points_lookup)

missing_rank = df_fixtures[df_fixtures['home_fifa_rank'].isna() |
                            df_fixtures['away_fifa_rank'].isna()]
if len(missing_rank) > 0:
    print(f"⚠️  {len(missing_rank)} fixtures have teams missing from df_rankings:")
    print(missing_rank[['home', 'away']].to_string())

# ── 4. WC historical filter (for diagnostics / separate analysis only) ───────
wc_mask       = df_played['tournament'] == 'FIFA World Cup'
df_wc_history = df_played[wc_mask].copy()
# NOTE: df_wc_history is for inspection/analysis — do NOT use as a separate
#       training set unless you're building a WC-only model. Keep df_played
#       as the single source of truth for training.

# ── 5. Sanity checks ─────────────────────────────────────────────────────────
print(f"✅ df_played rows              : {len(df_played):,}")
print(f"   ELO pseudo-rank null (home) : {df_played['home_elo_rank'].isna().sum()}")
print(f"   ELO pseudo-rank null (away) : {df_played['away_elo_rank'].isna().sum()}")
print(f"\n✅ df_fixtures rows            : {len(df_fixtures):,}")
print(f"   FIFA rank null (home)       : {df_fixtures['home_fifa_rank'].isna().sum()}")
print(f"   FIFA rank null (away)       : {df_fixtures['away_fifa_rank'].isna().sum()}")
print(f"\n✅ WC historical matches       : {len(df_wc_history):,}")
print(f"   Date range: {df_wc_history['date'].min().date()} → "
      f"{df_wc_history['date'].max().date()}")

⚠️  38 fixtures have teams missing from df_rankings:
            home       away
25   Ivory Coast    Curaçao
27       Curaçao    Ecuador
28       Germany    Curaçao
54     Argentina       Cuba
57        Jordan       Cuba
59          Cuba  Venezuela
72            1A         2B
73            1B         2A
74            1C         2D
75            1D         2C
76            1E         2F
77            1F         2E
78            1G         2H
79            1H         2G
80            1I         2J
81            1J         2I
82            1K         2L
83            1L         2K
84         3rd-1      3rd-2
85         3rd-3      3rd-4
86         3rd-5      3rd-6
87         3rd-7      3rd-8
88           W73        W74
89           W75        W76
90           W77        W78
91           W79        W80
92           W81        W82
93           W83        W84
94           W85        W86
95           W87        W88
96           W89        W90
97           W91        W92
98           W93       

---
## 5. Build the ML Feature Matrix

We now assemble the final training dataset. Each row = one match.

**Features per match:**

| Feature | Description |
|---------|-------------|
| `elo_diff` | Home ELO − Away ELO (before match) |
| `home_elo_before` | Home team ELO |
| `away_elo_before` | Away team ELO |
| `home_win_rate_5/10` | Home team win rate (last 5/10) |
| `away_win_rate_5/10` | Away team win rate (last 5/10) |
| `home_avg_gf/ga/gd_5/10` | Home team goal stats (last 5/10) |
| `away_avg_gf/ga/gd_5/10` | Away team goal stats (last 5/10) |
| `win_rate_diff_5/10` | Home win rate − Away win rate |
| `is_neutral` | Neutral venue (1/0) |
| `h2h_home_win_rate` | Historical H2H win rate for home team |
| `h2h_games` | Number of H2H matches played |

**Target:** `result` — 0 = Away win, 1 = Draw, 2 = Home win

In [24]:
def build_feature_matrix(df: pd.DataFrame, h2h_df: pd.DataFrame) -> pd.DataFrame:
    """
    Assemble the full feature matrix from a match DataFrame.
    Fully vectorized — no iterrows().
    Windows: 3, 5, 10
    """
    out = df.copy()

    # ── Ensure elo_diff exists ───────────────────────────────────────────────
    if 'elo_diff' not in out.columns:
        out['elo_diff'] = out['home_elo_before'] - out['away_elo_before']

    # ── Derived form features (all 3 windows) ────────────────────────────────
    for w in [3, 5, 10]:
        out[f'win_rate_diff_{w}'] = out[f'home_win_rate_{w}'] - out[f'away_win_rate_{w}']
        out[f'gd_diff_{w}']       = out[f'home_avg_gd_{w}']  - out[f'away_avg_gd_{w}']

    # ── Rank features ────────────────────────────────────────────────────────
    if 'home_elo_rank' in out.columns and 'away_elo_rank' in out.columns:
        out['rank_diff'] = out['home_elo_rank'] - out['away_elo_rank']

    # ── Neutral ground ───────────────────────────────────────────────────────
    out['is_neutral'] = out['neutral'].eq(True).astype(int)

 

    # ── Days since last match (per team) ─────────────────────────────────────
    out = out.sort_values('date').reset_index(drop=True)
    for side in ['home', 'away']:
        out[f'{side}_days_rest'] = (
            out.groupby(f'{side}_team')['date']
            .shift(1)
            .rsub(out['date'])
            .dt.days
        )

    # ── H2H features (single merge, both directions pre-stored) ─────────────
    out = out.merge(
        h2h_df[['team_a', 'team_b', 'games', 'a_wins', 'draws', 'b_wins',
                 'a_goals', 'b_goals', 'avg_goal_diff',
                 'a_win_rate', 'a_win_rate_weighted']].rename(columns={
            'team_a':               'home_team',
            'team_b':               'away_team',
            'games':                'h2h_games',
            'a_wins':               'h2h_home_wins',
            'draws':                'h2h_draws',
            'b_wins':               'h2h_away_wins',
            'a_goals':              'h2h_home_goals',
            'b_goals':              'h2h_away_goals',
            'avg_goal_diff':        'h2h_avg_goal_diff',
            'a_win_rate':           'h2h_home_win_rate',
            'a_win_rate_weighted':  'h2h_home_win_rate_w',
        }),
        on=['home_team', 'away_team'],
        how='left'
    )
    out['h2h_games'] = out['h2h_games'].fillna(0).astype(int)

    # ── Target label ─────────────────────────────────────────────────────────
    out['result'] = np.select(
        [out['home_score'] > out['away_score'],
         out['home_score'] == out['away_score']],
        [2, 1],
        default=0
    )

    # ── Select final columns ─────────────────────────────────────────────────
    feature_cols = [
        # ELO
        'home_elo_before', 'away_elo_before', 'elo_diff',
        # Rank
        'home_elo_rank', 'away_elo_rank', 'rank_diff',
        # Form — 3-match window
        'home_win_rate_3', 'away_win_rate_3', 'win_rate_diff_3',
        'home_avg_gf_3', 'home_avg_ga_3', 'home_avg_gd_3',
        'away_avg_gf_3', 'away_avg_ga_3', 'away_avg_gd_3', 'gd_diff_3',
        # Form — 5-match window
        'home_win_rate_5', 'away_win_rate_5', 'win_rate_diff_5',
        'home_avg_gf_5', 'home_avg_ga_5', 'home_avg_gd_5',
        'away_avg_gf_5', 'away_avg_ga_5', 'away_avg_gd_5', 'gd_diff_5',
        # Form — 10-match window
        'home_win_rate_10', 'away_win_rate_10', 'win_rate_diff_10',
        'home_avg_gf_10', 'home_avg_ga_10', 'home_avg_gd_10',
        'away_avg_gf_10', 'away_avg_ga_10', 'away_avg_gd_10', 'gd_diff_10',
        # H2H
        'h2h_games', 'h2h_home_wins', 'h2h_draws', 'h2h_away_wins',
        'h2h_home_goals', 'h2h_away_goals',
        'h2h_avg_goal_diff', 'h2h_home_win_rate', 'h2h_home_win_rate_w',
        # Context
        'is_neutral', 
        'home_days_rest', 'away_days_rest',
        # Target
        'result',
        # Metadata
        'date', 'home_team', 'away_team', 'tournament',
    ]

    final_cols = [c for c in feature_cols if c in out.columns]
    missing    = [c for c in feature_cols if c not in out.columns]
    if missing:
        print(f"⚠️  Columns not found, skipped: {missing}")

    return out[final_cols]


print('⚙️  Building ML feature matrix...')
df_features = build_feature_matrix(df_played, df_h2h)
print(f'✅ Feature matrix: {df_features.shape[0]:,} rows × {df_features.shape[1]} columns')

# Null audit
meta = ['date', 'home_team', 'away_team', 'tournament', 'result']
feat_only  = [c for c in df_features.columns if c not in meta]
null_rates = df_features[feat_only].isna().mean().sort_values(ascending=False)
print('\nNull rates per feature:')
print(null_rates[null_rates > 0].to_string())

⚙️  Building ML feature matrix...
✅ Feature matrix: 32,286 rows × 53 columns

Null rates per feature:
h2h_draws              0.874218
h2h_away_wins          0.874218
h2h_home_wins          0.874218
h2h_home_win_rate_w    0.874218
h2h_home_win_rate      0.874218
h2h_avg_goal_diff      0.874218
h2h_away_goals         0.874218
h2h_home_goals         0.874218
gd_diff_3              0.015982
win_rate_diff_3        0.015982
gd_diff_5              0.015982
win_rate_diff_5        0.015982
gd_diff_10             0.015982
win_rate_diff_10       0.015982
away_win_rate_3        0.010500
away_avg_gf_3          0.010500
away_avg_ga_5          0.010500
away_avg_gf_10         0.010500
away_avg_ga_10         0.010500
away_avg_gd_10         0.010500
away_avg_gd_5          0.010500
away_win_rate_10       0.010500
away_avg_gf_5          0.010500
away_win_rate_5        0.010500
away_avg_gd_3          0.010500
away_avg_ga_3          0.010500
home_days_rest         0.009880
away_days_rest         0.009633
ho

In [25]:
#display(df_features)

In [26]:
core_feats = ['home_elo_before', 'away_elo_before', 'home_win_rate_5', 'away_win_rate_5']
before = len(df_features)
df_features = df_features.dropna(subset=core_feats).reset_index(drop=True)
print(f'Dropped {before - len(df_features):,} rows with insufficient history')

meta = ['date', 'home_team', 'away_team', 'tournament', 'result']
feat_only = [c for c in df_features.columns if c not in meta]

# H2H rates have a meaningful no-data default — fill separately
h2h_rate_cols = ['h2h_home_win_rate', 'h2h_home_win_rate_w']
h2h_count_cols = ['h2h_games', 'h2h_home_wins', 'h2h_draws',
                   'h2h_away_wins', 'h2h_home_goals', 'h2h_away_goals']
other_feats = [c for c in feat_only
               if c not in h2h_rate_cols + h2h_count_cols]

df_features[h2h_rate_cols]  = df_features[h2h_rate_cols].fillna(0.5)
df_features[h2h_count_cols] = df_features[h2h_count_cols].fillna(0)
df_features[other_feats]    = df_features[other_feats].fillna(df_features[other_feats].median())

print(f'Null values remaining: {df_features[feat_only].isnull().sum().sum()}')
print(f'\nTarget distribution (result):')
vc = df_features['result'].value_counts().sort_index()
for label, name in zip([0,1,2], ['Away win', 'Draw', 'Home win']):
    pct = vc[label] / len(df_features) * 100
    print(f'  {name:12s}: {vc[label]:,}  ({pct:.1f}%)')

Dropped 516 rows with insufficient history
Null values remaining: 0

Target distribution (result):
  Away win    : 8,903  (28.0%)
  Draw        : 7,484  (23.6%)
  Home win    : 15,383  (48.4%)


In [27]:
# Save the full feature matrix
df_features.to_csv(PROC_DIR / 'features_all_matches.csv', index=False)
print(f'✅ Saved: features_all_matches.csv')

# Also save WC-only subset (optional — for WC-specific model training)
df_wc_features = df_features[df_features['tournament'] == 'FIFA World Cup'].reset_index(drop=True)
df_wc_features.to_csv(PROC_DIR / 'features_wc_only.csv', index=False)
print(f'✅ Saved: features_wc_only.csv  ({len(df_wc_features):,} WC matches)')

display(df_features[feat_only].describe().round(2))

✅ Saved: features_all_matches.csv
✅ Saved: features_wc_only.csv  (552 WC matches)


,home_elo_before,away_elo_before,elo_diff,home_elo_rank,away_elo_rank,rank_diff,home_win_rate_3,away_win_rate_3,win_rate_diff_3,home_avg_gf_3,...,h2h_draws,h2h_away_wins,h2h_home_goals,h2h_away_goals,h2h_avg_goal_diff,h2h_home_win_rate,h2h_home_win_rate_w,is_neutral,home_days_rest,away_days_rest
count,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,...,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00,31770.00
mean,152.62,141.26,11.36,17.01,17.41,-0.40,0.39,0.38,0.01,1.40,...,0.40,0.53,2.00,1.81,0.12,0.50,0.50,0.28,94.34,94.60
std,170.88,166.91,184.66,20.94,20.93,15.93,0.31,0.30,0.41,1.01,...,1.48,2.08,7.23,6.60,0.39,0.09,0.09,0.45,187.71,187.65
min,-534.57,-532.29,-1074.15,1.00,1.00,-105.00,0.00,0.00,-1.00,0.00,...,0.00,0.00,0.00,0.00,-6.00,0.00,0.00,0.00,0.00,0.00
25%,46.98,40.65,-92.01,3.00,3.00,-4.00,0.00,0.00,-0.33,0.67,...,0.00,0.00,0.00,0.00,0.11,0.50,0.50,0.00,7.00,8.00
50%,141.68,133.40,10.60,7.50,8.00,-1.00,0.33,0.33,0.00,1.33,...,0.00,0.00,0.00,0.00,0.11,0.50,0.50,0.00,39.00,41.00
75%,264.44,250.18,117.43,24.00,25.00,4.00,0.67,0.67,0.33,2.00,...,0.00,0.00,0.00,0.00,0.11,0.50,0.50,1.00,106.00,108.00
max,696.71,703.45,1086.73,132.00,130.00,110.00,1.00,1.00,1.00,21.00,...,13.00,21.00,68.00,68.00,7.00,1.00,1.00,1.00,6916.00,6600.00


---
## 6. Build WC 2026 Prediction Features

For each of the 72 group-stage fixtures, compute the same feature vector using **current** ELO and form (as of June 2026).

In [28]:
def get_team_current_form(team: str, df_history: pd.DataFrame,
                           windows: list[int] = [3, 5, 10]) -> dict:
    """Get a team's most recent rolling form stats — vectorized."""
    mask = (df_history['home_team'] == team) | (df_history['away_team'] == team)
    recent = df_history[mask].sort_values('date').reset_index(drop=True)

    if len(recent) == 0:
        results = {}
        for w in windows:
            results[f'win_rate_{w}'] = 0.5
            results[f'avg_gf_{w}']   = 1.0
            results[f'avg_ga_{w}']   = 1.0
            results[f'avg_gd_{w}']   = 0.0
        return results

    # Vectorized: resolve scores from team's perspective
    is_home  = recent['home_team'] == team
    gf = np.where(is_home, recent['home_score'], recent['away_score']).astype(float)
    ga = np.where(is_home, recent['away_score'], recent['home_score']).astype(float)
    wins = (gf > ga).astype(float)

    results = {}
    for w in windows:
        last_gf   = gf[-w:]
        last_ga   = ga[-w:]
        last_wins = wins[-w:]
        n = len(last_gf)
        results[f'win_rate_{w}'] = last_wins.mean() if n > 0 else 0.5
        results[f'avg_gf_{w}']   = last_gf.mean()   if n > 0 else 1.0
        results[f'avg_ga_{w}']   = last_ga.mean()   if n > 0 else 1.0
        results[f'avg_gd_{w}']   = results[f'avg_gf_{w}'] - results[f'avg_ga_{w}']
    return results


print('⚙️  Building features for all 72 WC 2026 group fixtures...')

wc26_group_fixtures = df_fixtures[df_fixtures['stage'] == 'Group'].copy()

# Pre-build H2H lookup dict for fast access
h2h_lookup = {
    (r['team_a'], r['team_b']): r
    for _, r in df_h2h.iterrows()
}

def get_h2h_row(home, away):
    r = h2h_lookup.get((home, away))
    if r is not None:
        return {
            'h2h_games':            int(r['games']),
            'h2h_home_wins':        int(r['a_wins']),
            'h2h_draws':            int(r['draws']),
            'h2h_away_wins':        int(r['b_wins']),
            'h2h_home_goals':       float(r['a_goals']),
            'h2h_away_goals':       float(r['b_goals']),
            'h2h_avg_goal_diff':    float(r['avg_goal_diff']),
            'h2h_home_win_rate':    float(r['a_win_rate']),
            'h2h_home_win_rate_w':  float(r['a_win_rate_weighted']),
        }
    return {
        'h2h_games': 0, 'h2h_home_wins': 0, 'h2h_draws': 0,
        'h2h_away_wins': 0, 'h2h_home_goals': 0.0, 'h2h_away_goals': 0.0,
        'h2h_avg_goal_diff': 0.0,
        'h2h_home_win_rate': 0.5, 'h2h_home_win_rate_w': 0.5,
    }

pred_rows = []
for _, fix in wc26_group_fixtures.iterrows():
    home, away = fix['home'], fix['away']

    h_form = get_team_current_form(home, df_played)
    a_form = get_team_current_form(away, df_played)

    h_elo  = current_elos.get(home, 1500)
    a_elo  = current_elos.get(away, 1500)
    h_rank = elo_long[elo_long['team'] == home]['elo_pseudo_rank'].iloc[-1] \
             if home in elo_long['team'].values else np.nan
    a_rank = elo_long[elo_long['team'] == away]['elo_pseudo_rank'].iloc[-1] \
             if away in elo_long['team'].values else np.nan

    h2h = get_h2h_row(home, away)

    row = {
        'match_id':   fix['match_id'],
        'date':       fix['date'],
        'group':      fix['group'],
        'home_team':  home,
        'away_team':  away,
        # ELO
        'home_elo_before': h_elo,
        'away_elo_before': a_elo,
        'elo_diff':        h_elo - a_elo,
        # Rank
        'home_elo_rank':   h_rank,
        'away_elo_rank':   a_rank,
        'rank_diff':       h_rank - a_rank if not (np.isnan(h_rank) or np.isnan(a_rank)) else np.nan,
        # Form — all 3 windows
        **{f'home_{k}': v for k, v in h_form.items()},
        **{f'away_{k}': v for k, v in a_form.items()},
        # Derived diffs
        **{f'win_rate_diff_{w}': h_form[f'win_rate_{w}'] - a_form[f'win_rate_{w}']
           for w in [3, 5, 10]},
        **{f'gd_diff_{w}': h_form[f'avg_gd_{w}'] - a_form[f'avg_gd_{w}']
           for w in [3, 5, 10]},
        # H2H — all columns
        **h2h,
        # Context
        'is_neutral':       1,
        'home_days_rest':   np.nan,  # unknown until schedule confirmed
        'away_days_rest':   np.nan,
    }
    pred_rows.append(row)

df_pred_features = pd.DataFrame(pred_rows)

# Verify column alignment with training matrix
train_feat_cols = [c for c in df_features.columns
                   if c not in ['date', 'home_team', 'away_team', 'tournament', 'result']]
pred_cols       = df_pred_features.columns.tolist()
meta_cols = ['date', 'home_team', 'away_team', 'tournament', 'result', 'match_id', 'group']

missing_in_pred = [c for c in train_feat_cols if c not in pred_cols]
extra_in_pred   = [c for c in pred_cols if c not in train_feat_cols
                   and c not in meta_cols]

if missing_in_pred:
    print(f"⚠️  In training but missing from prediction: {missing_in_pred}")
if extra_in_pred:
    print(f"⚠️  In prediction but not in training: {extra_in_pred}")

df_pred_features.to_csv(PROC_DIR / 'features_wc2026_group_stage.csv', index=False)
print(f'✅ Saved: features_wc2026_group_stage.csv  ({len(df_pred_features)} fixtures)')
display(df_pred_features[['home_team', 'away_team', 'elo_diff',
                           'home_win_rate_5', 'away_win_rate_5',
                           'h2h_home_win_rate', 'h2h_games']].head(10))

⚙️  Building features for all 72 WC 2026 group fixtures...
✅ Saved: features_wc2026_group_stage.csv  (72 fixtures)


,home_team,away_team,elo_diff,home_win_rate_5,away_win_rate_5,h2h_home_win_rate,h2h_games
0,Mexico,South Africa,254.205602,0.6,0.2,0.625,4
1,South Korea,Czech Republic,105.142900,0.6,0.6,0.500,3
2,Mexico,South Korea,77.525839,0.6,0.6,0.611,9
3,Czech Republic,South Africa,71.536863,0.6,0.2,0.500,1
4,Mexico,Czech Republic,182.668739,0.6,0.6,0.000,1
5,South Africa,South Korea,-176.679762,0.2,0.6,NaN,0
6,Canada,Bosnia and Herzegovina,243.671415,0.4,0.0,NaN,0
7,Qatar,Switzerland,-209.272865,0.0,0.2,1.000,1
8,Canada,Qatar,208.489936,0.4,0.0,1.000,1
9,Switzerland,Bosnia and Herzegovina,244.454344,0.2,0.0,0.000,1


---
## 7. Summary

In [29]:
import os

print('=' * 55)
print('FEATURE ENGINEERING COMPLETE')
print('=' * 55)
for f in sorted(PROC_DIR.glob('*.csv')):
    size_kb = os.path.getsize(f) / 1024
    print(f'  📄 {f.name:<45} {size_kb:>7.1f} KB')

print(f'''
Feature summary:
  Training rows  : {len(df_features):,}
  Feature columns: {len(feat_only)}
  WC 2026 fixtures ready for prediction: {len(df_pred_features)}
''')

FEATURE ENGINEERING COMPLETE
  📄 attack_defense_ratings.csv                       13.5 KB
  📄 confederation_strength.csv                        0.1 KB
  📄 elo_all_teams.csv                                10.7 KB
  📄 features_all_matches.csv                      12786.6 KB
  📄 features_wc2026_group_stage.csv                  30.3 KB
  📄 features_wc_only.csv                            224.1 KB
  📄 h2h_records.csv                                 105.0 KB
  📄 tournament_experience.csv                         1.1 KB
  📄 wc2026_scoreline_predictions.csv                 17.0 KB

Feature summary:
  Training rows  : 31,770
  Feature columns: 48
  WC 2026 fixtures ready for prediction: 72

